In [29]:
import asyncio
from agents import Agent, FileSearchTool, Runner, WebSearchTool, trace, set_trace_processors
import os
from dotenv import load_dotenv

from IPython.display import display, Markdown
import nest_asyncio
import pypandoc
import tempfile
import pandas as pd
from typing import Literal
from pydantic import BaseModel, Field
import weave
from weave.integrations.openai_agents.openai_agents import WeaveTracingProcessor
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [25]:
weave.init("openai-agents")
set_trace_processors([WeaveTracingProcessor()])

weave: Please login to Weights & Biases (https://wandb.ai/) to continue...
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=weave
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/Francisco_Reveriano/.netrc
wandb: Currently logged in as: francisco-reveriano-1 (francisco-reveriano-1-mckinsey-company) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
weave: Logged in as Weights & Biases user: francisco-reveriano-1.
weave: View Weave data at https://wandb.ai/francisco-reveriano-1-mckinsey-company/openai-a

## Prepare Agent

In [26]:
federal_tax_agent = Agent(
    name="Federal_Tax_Agent",
    instructions="You are an expert file-search tax agent for federal tax documents.",
    output_type=str,
        model="o3",
        tools=[
            FileSearchTool(
                max_num_results=40,
                vector_store_ids=["vs_683dca6814108191a83ccb974fff986d"],
                include_search_results=True,
            )
        ],
)

illinois_tax_agent = Agent(
    name="Illinois_Tax_Agent",
    instructions="You are an expert file-search tax agent for the state of Illinois",
    output_type=str,
        model="o3",
        tools=[
            FileSearchTool(
                max_num_results=40,
                vector_store_ids=["vs_6841d1063e788191aed6d433023b92e5"],
                include_search_results=True,
            )
        ],
)

TaxAgentPrompt = '''
        You are an expert tax advisor. You thoroughly read the document and provide a detailed answer to the user.
        "If asked for tax details regarding Federal or State taxes, you call the relevant tools.
'''
TaxAgent = Agent(
        name="TaxAgent",
        instructions=TaxAgentPrompt,
        output_type=str,
        model="o3",
        tools=[
            federal_tax_agent.as_tool(
                tool_name="Federal_Tax_Agent",
                tool_description="Use this tool to search for federal tax documents.",
            ),
            illinois_tax_agent.as_tool(
                tool_name="Illinois_Tax_Agent",
                tool_description="Use this tool to search for the state of Illinois specific tax documents.",
            )
        ],
    )

In [27]:
message = "I am 55 and will retire from McKinsey by 60, and live in Illinois, what steps should I take to optimize my situation?"
nest_asyncio.apply()
result = await Runner.run(
            TaxAgent, message
        )

weave: 🍩 https://wandb.ai/francisco-reveriano-1-mckinsey-company/openai-agents/r/call/01974130-537c-77e1-8af6-efc56550399b


In [40]:
result.final_output

'Below is a practical, Illinois-specific “playbook” for the next five-plus years and the first phase of retirement.  The steps are grouped by topic so you can turn them into a checklist and calendar.\n\n----------------------------------------------------------------\nA.  Maximize the “last big funding window” (ages 55-60)\n----------------------------------------------------------------\n1.  Employer plan catch-up contributions  \n   • 401(k) / 403(b) / most 457(b): Regular limit is $23,000 for 2024; age-50+ catch-up lets you add another $7,500, for a total of $30,500.  \n   • Verify that McKinsey’s plan actually offers catch-ups and, if you have access to both traditional and Roth buckets, decide each year which to use (see Section C on bracket management).  \n\n2.  IRA catch-ups  \n   • You can put up to $8,000 ($7,000 + $1,000 catch-up) into a Traditional or Roth IRA for 2024.  \n   • If your income is too high for a direct Roth, use the “back-door” Roth (non-deductible Traditional

In [28]:
display(Markdown(result.final_output))

Below is a practical, Illinois-specific “playbook” for the next five-plus years and the first phase of retirement.  The steps are grouped by topic so you can turn them into a checklist and calendar.

----------------------------------------------------------------
A.  Maximize the “last big funding window” (ages 55-60)
----------------------------------------------------------------
1.  Employer plan catch-up contributions  
   • 401(k) / 403(b) / most 457(b): Regular limit is $23,000 for 2024; age-50+ catch-up lets you add another $7,500, for a total of $30,500.  
   • Verify that McKinsey’s plan actually offers catch-ups and, if you have access to both traditional and Roth buckets, decide each year which to use (see Section C on bracket management).  

2.  IRA catch-ups  
   • You can put up to $8,000 ($7,000 + $1,000 catch-up) into a Traditional or Roth IRA for 2024.  
   • If your income is too high for a direct Roth, use the “back-door” Roth (non-deductible Traditional IRA → same-day conversion).  

3.  Health Savings Account (HSA) “extra” contributions  
   • You can add +$1,000 catch-up once you are 55, bringing 2024 family HSA space to $9,300.  
   • Pay current medical expenses out of pocket and let the HSA grow tax-free for retirement health costs.

----------------------------------------------------------------
B.  Map out your cash-flow timeline
----------------------------------------------------------------
Age 55–59½  
• 401(k) “Rule of 55.”  Because you will separate from service in the calendar year you turn 60, any distributions you take from that 401(k) can avoid the 10 % early-withdrawal penalty, even before 59½.  (IRAs do NOT get this break.)  
• Decide whether you need the flexibility; if you think you might, keep at least part of the balance in the McKinsey plan until 59½.

Age 59½  
• Penalty disappears for all retirement accounts.  This is the earliest “safe” date to draw from IRAs to bridge income needs, do Roth conversions and/or large charitable gifts funded from IRAs without a 10 % penalty.

Age 60–Medicare (65)  
• Health insurance: price COBRA from McKinsey vs. ACA marketplace plans; HSA dollars can pay COBRA or Medicare premiums tax-free.  
• Social Security: earliest widow(er)’s benefits start at 60, otherwise it is usually best to wait (see Section E).

----------------------------------------------------------------
C.  Lower lifetime federal tax through bracket management
----------------------------------------------------------------
1.  Identify your likely marginal brackets:  
   • Working years to 60 → probably 32 %–35 % federal.  
   • Early-retirement gap (60–70) → may drop to 22 %–24 %.  
   • RMD years (73+) + Social Security → bracket may rise again.

2.  Roth conversion sweet spot  
   • Convert just enough each gap-year to “fill” the 22 % or 24 % bracket but stay below Medicare IRMAA surcharges where possible.  
   • Pay the tax with taxable savings, not with money withheld from the conversion itself.

3.  Capital-gain harvesting  
   • If taxable income falls into the 0 % LTCG bracket (up to $94,050 MFJ in 2024), realize gains deliberately.  Illinois taxes capital gains, so you still owe 4.95 %, but a federal 0 % rate is hard to beat.

----------------------------------------------------------------
D.  Play Illinois rules to your advantage
----------------------------------------------------------------
• Illinois does NOT tax:  
  – Distributions from qualified 401(k)/403(b)/457 plans,  
  – Traditional or Roth IRA withdrawals,  
  – Private or public pensions,  
  – Social Security.  
  (All are subtracted on IL-1040, Line 5.)  
• Result: once you retire, almost all of your retirement-account income will be state-tax-free.  Therefore:  
  – The state cost of Roth vs. Traditional decisions is essentially zero; focus on federal brackets.  
  – If you move out of Illinois in retirement, model the impact carefully—many states do tax these items.

----------------------------------------------------------------
E.  Social Security optimization
----------------------------------------------------------------
• Each year you delay between 62 and 70 increases your benefit 7–8 % (plus COLAs).  
• Because Illinois doesn’t tax it, deferral is even more attractive.  
• Typical strategy for high earners: live on 401(k)/IRA withdrawals and Roth conversions 60–70, claim Social Security at 70.

----------------------------------------------------------------
F.  Pension or other McKinsey deferred-comp decisions
----------------------------------------------------------------
• If you have a cash-balance pension or non-qualified deferred comp (NQDC) plan, compare lump-sum vs. annuity payouts and scheduled distributions.  
• Aim to receive lump-sum/NQDC in your early-retirement low-tax window rather than while still on salary.  
• Confirm whether any portion is exempt from Illinois tax (qualified pension = exempt; NQDC usually taxed).

----------------------------------------------------------------
G.  Prepare for Required Minimum Distributions (RMDs)
----------------------------------------------------------------
• Your first RMD year is the year you turn 73; you may delay only the first one until April 1 of the next year, which would bunch two RMDs in that year.  
• Rolling old 401(k)s into one IRA (or into the active plan if still working) simplifies future RMD tracking.  
• Consider partial Roth conversions each year until the projected IRA balance at 73 lines up with your target tax bracket.

----------------------------------------------------------------
H.  Charitable strategies
----------------------------------------------------------------
1.  Donor-Advised Fund (DAF) “bunching” while working  
   • Front-load multiple years of donations into a DAF to exceed the itemized-deduction threshold in high-income years, then gift from the DAF later.

2.  Qualified Charitable Distributions (QCDs) after age 70½  
   • Up to $105,000/year goes directly from IRA to charities and counts toward RMD without hitting AGI.

----------------------------------------------------------------
I.  Insurance, health and long-term-care
----------------------------------------------------------------
• Life: Re-evaluate how much you need once salary stops.  
• Disability: Usually drop coverage at retirement.  
• Long-term-care: Pricing is best in your mid-50s; weigh premiums vs. self-funding from portfolio.  
• Umbrella liability: Make sure at least $1-2 million remains in force.

----------------------------------------------------------------
J.  Estate and beneficiary clean-up
----------------------------------------------------------------
• Update wills, powers-of-attorney and health-care directives.  
• Check that 401(k), IRA and insurance beneficiaries are correct; beneficiary designations override wills.  
• If you have significant Roth assets, name individual beneficiaries (not the estate) so they can use the 10-year stretch.

----------------------------------------------------------------
K.  Relocation and property-tax planning
----------------------------------------------------------------
• Illinois property taxes are high; if you plan to move after retirement, weigh housing costs vs. the benefit of Illinois’ retirement-income tax break.  
• Downsizing or moving to a lower-tax county can fund part of long-term-care or travel goals.

----------------------------------------------------------------
Action calendar (sample)
----------------------------------------------------------------
• Now:  
  – Elect maximum 401(k) + catch-up.  
  – Open/fully fund HSA and Roth/Back-door Roth.  
  – Price LTC insurance.  
• 59½ birthday:  
  – Re-visit withdrawal plan; decide whether to roll the 401(k) to an IRA or keep it for Rule-of-55 access.  
• Year you retire (age 60):  
  – Start gap-year Roth conversion plan.  
  – Choose COBRA vs. ACA insurance.  
• Age 62:  
  – Run Social-Security-claiming analysis but likely defer.  
• Age 70:  
  – File for maximum Social Security.  
• Age 73:  
  – First RMD (unless converting enough to eliminate it).  Consider QCD eligibility at 70½+.

----------------------------------------------------------------
Key takeaways
----------------------------------------------------------------
1. Your highest-impact levers between 55 and 60 are (a) max funding—including catch-ups and HSA—while salary is strong, and (b) laying groundwork for Roth conversions once income drops.  
2. Illinois’ complete exemption of qualified retirement income means state taxes are not a constraint; focus on federal bracket-management and IRMAA thresholds.  
3. Execute a multi-year plan: fund, convert, and draw in the right sequence, while keeping an eye on health coverage and estate documents.

Work through this list with your tax professional and financial planner annually; small course-corrections (e.g., conversion amounts, DAF vs. cash gifts, marketplace premium subsidies) can save thousands every year and tens of thousands over your lifetime.

## Save Document

In [1]:
from src.Markdown_to_PDF_Document import *
markdown_text = result.final_output
output_pdf_path = "../Data/Final/TaxReport.docx"
markdown_to_docx(markdown_text, output_pdf_path)

✅ Word document successfully generated at: ../Data/Final/TaxReport.docx
